# Weight-trajectory PCA on a toy task

Train a small tanh MLP on two moons with plain SGD for 1,000 steps, then:

1. record the flat weight vector $w_i$ after every step $i = 0,\dots,999$,
2. form the trailing average over the last 50 steps, $\bar{w}_i = \mathrm{mean}(w_{i-49},\dots,w_i)$ (partial window for $i<49$),
3. fit PCA to the 1,000 averaged points $\{\bar{w}_i\}$,
4. project **both** the averaged path and the raw noisy SGD path onto the top-3 PCs — and, as a variant, onto the top-2 PCs only.

All figures are saved to `diff_plots/`.

In [ ]:
import os

import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils import parameters_to_vector
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_rgba
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d.art3d import Line3DCollection

SEED = 0
N_STEPS = 1000
WINDOW = 50
N_DATA = 600
DATA_NOISE = 0.18
BATCH_SIZE = 16
LR = 0.2
HIDDEN = 32
PLOT_DIR = "diff_plots"

os.makedirs(PLOT_DIR, exist_ok=True)
torch.set_num_threads(2)  # tiny model; don't grab the whole shared node

# palette (light mode)
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
BASE = "#c3c2b7"
BLUE = "#2a78d6"
ORANGE = "#eb6834"
RAW = "#c3c2b7"      # recessive gray for the raw SGD path
DARKEST = "#0d366b"  # ramp end, also the end-of-training marker
RAMP = LinearSegmentedColormap.from_list(
    "blue_ramp", ["#86b6ef", "#3987e5", "#256abf", "#104281", "#0d366b"]
)

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": BASE,
    "axes.labelcolor": INK2,
    "axes.titlecolor": INK,
    "axes.titlesize": 12,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "text.color": INK,
    "font.size": 11,
    "legend.frameon": False,
    "figure.dpi": 110,
})


def trailing_mean(arr, window):
    """Mean over the trailing `window` entries; partial window at the start."""
    a = np.asarray(arr, dtype=np.float64)
    cs = np.cumsum(a, axis=0)
    out = np.empty_like(cs)
    for i in range(len(a)):
        lo = max(0, i - window + 1)
        out[i] = (cs[i] - (cs[lo - 1] if lo > 0 else 0.0)) / (i - lo + 1)
    return out


def save_fig(fig, name):
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("saved", path)

In [ ]:
rng = np.random.default_rng(SEED)
n0 = N_DATA // 2
n1 = N_DATA - n0
t0 = rng.uniform(0.0, np.pi, n0)
t1 = rng.uniform(0.0, np.pi, n1)
X_np = np.concatenate([
    np.stack([np.cos(t0), np.sin(t0)], axis=1),
    np.stack([1.0 - np.cos(t1), 0.5 - np.sin(t1)], axis=1),
]) + rng.normal(0.0, DATA_NOISE, (N_DATA, 2))
y_np = np.concatenate([np.zeros(n0, np.int64), np.ones(n1, np.int64)])
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np)

fig, ax = plt.subplots(figsize=(6.2, 5.0), constrained_layout=True)
for cls, col in [(0, BLUE), (1, ORANGE)]:
    m = y_np == cls
    ax.scatter(X_np[m, 0], X_np[m, 1], s=14, color=col, alpha=0.85,
               linewidths=0, label=f"class {cls}")
ax.set_aspect("equal")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend(loc="upper right")
ax.set_title(f"Toy task: two moons ({N_DATA} points, noise {DATA_NOISE})")
save_fig(fig, "toy_task_data.png")
plt.show()

In [ ]:
torch.manual_seed(SEED)
model = nn.Sequential(
    nn.Linear(2, HIDDEN), nn.Tanh(),
    nn.Linear(HIDDEN, HIDDEN), nn.Tanh(),
    nn.Linear(HIDDEN, 2),
)
n_params = sum(p.numel() for p in model.parameters())

opt = torch.optim.SGD(model.parameters(), lr=LR)  # plain SGD, no momentum
loss_fn = nn.CrossEntropyLoss()
batch_rng = np.random.default_rng(SEED + 1)

W = np.empty((N_STEPS, n_params))  # W[i] = weights after step i
losses = np.empty(N_STEPS)
for i in range(N_STEPS):
    idx = batch_rng.choice(N_DATA, BATCH_SIZE, replace=False)
    loss = loss_fn(model(X[idx]), y[idx])
    opt.zero_grad()
    loss.backward()
    opt.step()
    W[i] = parameters_to_vector(model.parameters()).detach().numpy()
    losses[i] = loss.item()

with torch.no_grad():
    logits = model(X)
    full_loss = loss_fn(logits, y).item()
    acc = (logits.argmax(1) == y).float().mean().item()
print(f"{n_params} parameters, {N_STEPS} SGD steps recorded")
print(f"final full-data loss {full_loss:.4f}, accuracy {acc:.1%}")

In [ ]:
loss_avg = trailing_mean(losses, WINDOW)

fig, ax = plt.subplots(figsize=(7.6, 4.2), constrained_layout=True)
ax.plot(losses, color=RAW, linewidth=0.8, label="per-step minibatch loss")
ax.plot(loss_avg, color=BLUE, linewidth=2.0, label=f"trailing {WINDOW}-step mean")
ax.set_yscale("log")
ax.set_xlabel("step")
ax.set_ylabel("cross-entropy loss")
ax.legend()
ax.set_title(f"Training loss — final full-data loss {full_loss:.3f}, accuracy {acc:.1%}")
save_fig(fig, "training_loss.png")
plt.show()

In [ ]:
W_avg = trailing_mean(W, WINDOW)  # w_avg_i, i = 0 ... 999

mu = W_avg.mean(axis=0)
U, S, Vt = np.linalg.svd(W_avg - mu, full_matrices=False)
evr = S**2 / np.sum(S**2)

P3 = Vt[:3]
P2 = Vt[:2]
# same center + basis for both paths, so they live in the same coordinates
Z3_avg = (W_avg - mu) @ P3.T
Z3_raw = (W - mu) @ P3.T
Z2_avg = (W_avg - mu) @ P2.T
Z2_raw = (W - mu) @ P2.T

assert np.allclose(Z2_avg, Z3_avg[:, :2]) and np.allclose(Z2_raw, Z3_raw[:, :2])
print("PCA components are nested: top-2 projection == first two coords of top-3")
print("explained variance ratio (top 5):", np.array2string(evr[:5], precision=4))
print(f"cumulative: top-2 = {evr[:2].sum():.1%}, top-3 = {evr[:3].sum():.1%}")

In [ ]:
k = 10
fig, ax = plt.subplots(figsize=(7.0, 4.0), constrained_layout=True)
ax.bar(range(1, k + 1), evr[:k], color=[BLUE] * 3 + [BASE] * (k - 3), width=0.62)
for j in range(3):
    ax.text(j + 1, evr[j] + 0.015, f"{evr[j]:.1%}", ha="center", color=INK2, fontsize=10)
ax.set_xticks(range(1, k + 1), [f"PC{j}" for j in range(1, k + 1)])
ax.set_ylabel("explained variance ratio")
ax.set_ylim(0, evr[0] * 1.16)
ax.grid(axis="x", visible=False)
ax.set_title(f"PCA of averaged path — kept PCs (blue) explain {evr[:3].sum():.1%} "
             f"(top-2: {evr[:2].sum():.1%})")
save_fig(fig, "pca_explained_variance.png")
plt.show()

In [ ]:
def path_limits(Zs, i, pad=0.06):
    v = np.concatenate([Z[:, i] for Z in Zs])
    lo, hi = v.min(), v.max()
    m = pad * (hi - lo)
    return lo - m, hi + m


def ramp_line_2d(ax, Za, i, j):
    pts = Za[:, [i, j]].reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, cmap=RAMP, norm=Normalize(0, N_STEPS - 1),
                        linewidths=2.0, capstyle="round", zorder=3)
    lc.set_array(np.arange(N_STEPS - 1) + 0.5)
    ax.add_collection(lc)
    return lc


def mark_ends_2d(ax, Za, i, j):
    ax.scatter(*Za[0, [i, j]], s=65, facecolor=SURFACE, edgecolor=INK,
               linewidth=1.4, zorder=5)
    ax.scatter(*Za[-1, [i, j]], s=150, marker="*", facecolor=DARKEST,
               edgecolor=SURFACE, linewidth=0.7, zorder=5)
    ax.annotate("start", Za[0, [i, j]], xytext=(6, 6),
                textcoords="offset points", color=INK2, fontsize=9)
    ax.annotate("end", Za[-1, [i, j]], xytext=(6, 6),
                textcoords="offset points", color=INK2, fontsize=9)


def style_cbar(cb):
    cb.set_label("training step", color=INK2)
    cb.outline.set_edgecolor(BASE)
    cb.ax.tick_params(color=MUTED, labelcolor=MUTED)


legend_handles = [
    Line2D([], [], color=RAW, linewidth=1.2, label="raw SGD path $w_i$"),
    Line2D([], [], color=BLUE, linewidth=2.4,
           label=rf"trailing {WINDOW}-step average $\bar{{w}}_i$"),
]

In [ ]:
fig = plt.figure(figsize=(8.8, 7.2))
ax = fig.add_subplot(projection="3d")

ax.plot(Z3_raw[:, 0], Z3_raw[:, 1], Z3_raw[:, 2],
        color=RAW, linewidth=0.7, alpha=0.6, zorder=1)
pts = Z3_avg.reshape(-1, 1, 3)
segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
lc = Line3DCollection(segs, cmap=RAMP, norm=Normalize(0, N_STEPS - 1), linewidths=2.4)
lc.set_array(np.arange(N_STEPS - 1) + 0.5)
ax.add_collection3d(lc)

ax.scatter(*Z3_avg[0], s=70, facecolor=SURFACE, edgecolor=INK,
           linewidth=1.4, depthshade=False)
ax.scatter(*Z3_avg[-1], s=180, marker="*", facecolor=DARKEST,
           edgecolor=SURFACE, linewidth=0.7, depthshade=False)
ax.text(*Z3_avg[0], "  start", color=INK2, fontsize=9)
ax.text(*Z3_avg[-1], "  end", color=INK2, fontsize=9)

for i, setter in enumerate([ax.set_xlim, ax.set_ylim, ax.set_zlim]):
    setter(path_limits([Z3_raw, Z3_avg], i))
ax.set_xlabel(f"PC1 ({evr[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({evr[1]:.1%} var)")
ax.set_zlabel(f"PC3 ({evr[2]:.1%} var)")
for axis3 in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis3.set_pane_color(to_rgba(SURFACE))
    try:
        axis3._axinfo["grid"].update(color=GRID, linewidth=0.7)
    except Exception:
        pass
ax.view_init(elev=22, azim=-58)
ax.set_title("Weight trajectory in the top-3 PCA space of the averaged path", pad=14)
ax.legend(handles=legend_handles, loc="upper left")
style_cbar(fig.colorbar(lc, ax=ax, shrink=0.55, pad=0.06))
save_fig(fig, "pca3_trajectory_3d.png")
plt.show()

In [ ]:
pairs = [(0, 1), (0, 2), (1, 2)]
fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.6), constrained_layout=True)
for ax, (i, j) in zip(axes, pairs):
    ax.plot(Z3_raw[:, i], Z3_raw[:, j], color=RAW, linewidth=0.6, alpha=0.6, zorder=1)
    lc = ramp_line_2d(ax, Z3_avg, i, j)
    mark_ends_2d(ax, Z3_avg, i, j)
    ax.set_xlim(path_limits([Z3_raw, Z3_avg], i))
    ax.set_ylim(path_limits([Z3_raw, Z3_avg], j))
    ax.set_xlabel(f"PC{i + 1} ({evr[i]:.1%})")
    ax.set_ylabel(f"PC{j + 1} ({evr[j]:.1%})")
axes[0].legend(handles=legend_handles, loc="upper left", fontsize=9)
fig.suptitle("Pairwise views of the top-3 PCA projection", fontsize=13,
             fontweight="bold", color=INK)
style_cbar(fig.colorbar(lc, ax=axes, shrink=0.92, pad=0.015))
save_fig(fig, "pca3_pairwise.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 6.4), constrained_layout=True)
ax.plot(Z2_raw[:, 0], Z2_raw[:, 1], color=RAW, linewidth=0.7, alpha=0.6, zorder=1)
lc = ramp_line_2d(ax, Z2_avg, 0, 1)
mark_ends_2d(ax, Z2_avg, 0, 1)
ax.set_xlim(path_limits([Z2_raw, Z2_avg], 0))
ax.set_ylim(path_limits([Z2_raw, Z2_avg], 1))
ax.set_xlabel(f"PC1 ({evr[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({evr[1]:.1%} var)")
ax.legend(handles=legend_handles, loc="best", fontsize=9)
ax.set_title(f"Top-2 PCA variant — {evr[:2].sum():.1%} of averaged-path variance")
style_cbar(fig.colorbar(lc, ax=ax, pad=0.02))
save_fig(fig, "pca2_trajectory.png")
plt.show()

## Reading the plots

- The trailing average strips most minibatch noise, so $\bar{w}_i$ traces the smooth "backbone" of the trajectory, while the raw SGD path jitters around it.
- PCA is fit on the averaged path only, but the raw path is projected with the same center and basis, so both live in the same coordinates.
- PCA components are nested: the top-2 variant uses exactly the first two of the top-3 directions (asserted above), so the top-2 figure equals the PC1–PC2 panel with the PC3 information discarded.